- Cómputo final
    - (consulta 1)[cod pr]
    - (consulta 1)[desc pr]
    - (consulta 1)[num trenes]
    - (consulta 1)suma([CapturaVía])
    - (consulta 1)suma([CapturaVía])/[num trenes]
    - (consulta 1)suma([CoincideVía])
    - (consulta 1)suma([CoincideVía])/[num trenes]

- Detalle tren
    - (consulta 1)[ViaEstNum]
    - (consulta 1)[vía real de estacionamiento]
    - 


In [1]:
import xml.etree.ElementTree as ET
from collections import Counter
from datetime import datetime, timedelta
from pathlib import Path
from typing import List, Union

import numpy as np
import pandas as pd
import plotly.graph_objects as go
import regex
import requests
import yaml
from plotly.colors import qualitative
from tqdm.auto import tqdm

from src.api.api import GraylogAPIProcessor
from src.processor.log_procesor import LogProcessor
from src.utils import formatTimedelta, guardarExcel, parallelizeFunction

In [12]:
def sortStrNumbers(elements: list[str]):
    """
    Ordena una lista de strings que contiene números por número.
    """
    elements = [el for el in elements if regex.search("\d+", el)]
    return sorted(elements, key=lambda x: int(regex.search("\d+", x).group()))


def argSortStrNumbers(elements: list[str]):
    """
    Ordena una lista de strings que contiene números por número.
    """
    elements = [int(v.group()) for el in elements if (v := regex.search("\d+", el))]
    return np.argsort(elements)


def loadDetalles(fname: Path, sheet_name: str):
    if sheet_name == "Detalle Tren":
        skip = 1
    elif sheet_name == "Computo Estación":
        skip = 0
    elif sheet_name == "Computo Estación (Agrupadas)":
        skip = 4
    df = pd.read_excel(
        fname, sheet_name=sheet_name, engine="openpyxl", skiprows=skip
    ).dropna(axis=1, how="all")
    if sheet_name == "Computo Estación":
        fecha = pd.to_datetime(df.iloc[0, 0].split()[-1], dayfirst=True)
        df = pd.DataFrame(df.values[3:], columns=df.values[2])
        product = (
            pd.read_excel(
                fname, sheet_name="Computo Ámbito", engine="openpyxl", skiprows=0
            )
            .dropna(axis=1, how="all")
            .iloc[2, 5]
        )
        df["Fecha"] = fecha
        df["Producto"] = product
    return df

In [4]:
# product = (
#     pd.read_excel(fname, sheet_name="Computo Ámbito", engine="openpyxl", skiprows=0)
#     .dropna(axis=1, how="all")
#     .iloc[2, 5]
# )

In [13]:
# fname = Path(
#     "data/seg_vias/Vias Tren Compara Vias Parada Comercial - Ayer - Total_1053_5693436875483918954.xlsx"
# )
df_cha = []
fdir = Path("data/seg_vias")
# for fname in fdir.iterdir():
for fname in fdir.rglob("*"):
    if not fname.is_file():
        continue
    df = loadDetalles(fname, "Detalle Tren")
    df_cha.append(
        df.loc[
            (df["Cod. Est"] == 17000)
            # & (df["Fuente de Datos Vía Real de Estacionamiento"] == "CTC")
            ,
        ]
    )
df_cha = (
    pd.concat(df_cha)
    .rename(
        columns={
            "Tren": "Tren",
            "Fecha Origen Tren": "FechaOrigen",
            "Fecha Teorica Mov_PR": "FechaTeórica",
            "Cod. Est": "Código",
            "Estación": "Nombre",
            "Vía Teórica": "VíaPlanificada",
            "Vía Real de Estacionamiento": "VíaReal",
            "Fuente de Datos Vía Real de Estacionamiento": "FuenteVía",
        }
    )
    .sort_values(by=["FechaOrigen", "VíaPlanificada"])
)

c:\Users\jose.espinosa\Documents\LogProcess\.venv\Lib\site-packages\openpyxl\styles\stylesheet.py:226: UserWarning: Workbook contains no default style, apply openpyxl's default
  warn("Workbook contains no default style, apply openpyxl's default")
c:\Users\jose.espinosa\Documents\LogProcess\.venv\Lib\site-packages\openpyxl\styles\stylesheet.py:226: UserWarning: Workbook contains no default style, apply openpyxl's default
  warn("Workbook contains no default style, apply openpyxl's default")
c:\Users\jose.espinosa\Documents\LogProcess\.venv\Lib\site-packages\openpyxl\styles\stylesheet.py:226: UserWarning: Workbook contains no default style, apply openpyxl's default
  warn("Workbook contains no default style, apply openpyxl's default")
c:\Users\jose.espinosa\Documents\LogProcess\.venv\Lib\site-packages\openpyxl\styles\stylesheet.py:226: UserWarning: Workbook contains no default style, apply openpyxl's default
  warn("Workbook contains no default style, apply openpyxl's default")
c:\Users

In [15]:
# df["Estación"].str.split("-")

In [16]:
days = df_cha["FechaTeórica"].unique()
vias = df_cha["VíaPlanificada"].unique()
info = []
for d in days:
    for v in vias:
        subset = df_cha.loc[
            (df_cha["FechaTeórica"] == d) & (df_cha["VíaPlanificada"] == v),
            ["VíaPlanificada", "VíaReal"],
        ]
        if subset.empty:
            continue
        info.append(
            (
                d,
                int(v),
                subset.apply(
                    lambda x: x["VíaPlanificada"] == x["VíaReal"], axis=1
                ).sum()
                / len(subset)
                * 100,
            )
        )
coincidencias = pd.DataFrame(info, columns=["Fecha", "Vía", "Coincidencia"])
coincidencias["Día"] = coincidencias["Fecha"].dt.strftime("%Y-%m-%d")
coincidencias["Coincidencia (%)"] = coincidencias["Coincidencia"].apply(
    lambda x: f"{x:.3f}%"
)

In [19]:
df_estaciones = []
fdir = Path("data/seg_vias")
# for fname in fdir.iterdir():
for fname in fdir.rglob("*"):
    if not fname.is_file():
        continue
    df = loadDetalles(fname, "Computo Estación")
    if not df["Producto"].iloc[0] == "CERCANIAS":
        continue
    df[["Código", "Nombre"]] = df["Estación"].str.split("-", n=1, expand=True).values
    df[["Código", "Nombre"]] = df[["Código", "Nombre"]].map(str.strip)
    df_estaciones.append(
        df.loc[
            (df["Código"].isin(["72305", "65000", "70103", "05513", "51050"]))
            # & (df["Fuente de Datos Vía Real de Estacionamiento"] == "CTC")
            ,
        ]
    )
df_estaciones = pd.concat(df_estaciones).sort_values(by="Fecha")

for c in df_estaciones.columns:
    if "%" in c:
        df_estaciones[c] = df_estaciones[c] * 100

days = df_estaciones["Fecha"].unique()
station = df_estaciones["Código"].unique()
df_estaciones["Día"] = df_estaciones["Fecha"].dt.strftime("%Y-%m-%d")
df_estaciones["Coincidencia (%)"] = df_estaciones["% Coincide Vía"].apply(
    lambda x: f"{x:.3f}%"
)

c:\Users\jose.espinosa\Documents\LogProcess\.venv\Lib\site-packages\openpyxl\styles\stylesheet.py:226: UserWarning: Workbook contains no default style, apply openpyxl's default
  warn("Workbook contains no default style, apply openpyxl's default")
c:\Users\jose.espinosa\Documents\LogProcess\.venv\Lib\site-packages\openpyxl\styles\stylesheet.py:226: UserWarning: Workbook contains no default style, apply openpyxl's default
  warn("Workbook contains no default style, apply openpyxl's default")
c:\Users\jose.espinosa\Documents\LogProcess\.venv\Lib\site-packages\openpyxl\styles\stylesheet.py:226: UserWarning: Workbook contains no default style, apply openpyxl's default
  warn("Workbook contains no default style, apply openpyxl's default")
c:\Users\jose.espinosa\Documents\LogProcess\.venv\Lib\site-packages\openpyxl\styles\stylesheet.py:226: UserWarning: Workbook contains no default style, apply openpyxl's default
  warn("Workbook contains no default style, apply openpyxl's default")
c:\Users

In [22]:
# df_estaciones[df_estaciones["Código"] == "51050"].sort_values(by="Día")

In [23]:
# df_estaciones.head()

In [24]:
def setLayout(groupclick: str, title: str = "", category_map: dict = dict()):
    """
    groupclick: {"togglegroup", "toggleitem"}
    """
    if category_map:
        tickmode = "array"
        tickvals = list(category_map.values())
        ticktext = list(category_map.keys())
    else:
        tickmode = "auto"
        tickvals = None
        ticktext = None
    layout = go.Layout(
        showlegend=True,
        hoverlabel=dict(bgcolor="white", font_size=16, font_family="consolas"),
        yaxis=dict(
            # categoryorder="array",
            # categoryarray=categoryarray,
            tickmode=tickmode,
            tickvals=tickvals,
            ticktext=ticktext,
            autorange=True,
            fixedrange=False,
        ),
        xaxis=dict(
            rangeslider=dict(
                visible=True,
                thickness=0.1,
            ),
            type="date",
            autorange=True,
            fixedrange=False,
        ),
        legend=dict(
            groupclick=groupclick,
            # groupclick="togglegroup",
            yanchor="middle",
            y=0.5,
        ),
        title=title,
        modebar=dict(add="v1hovermode"),
        hoverdistance=20,
        # hovermode="x unified",
    )
    return layout

In [25]:
traces = []
hover_cols = ["Vía", "Día", "Coincidencia (%)"]
flen = max([len(c) for c in hover_cols]) + 2
fd_len = min(
    coincidencias[hover_cols]
    .fillna("")
    .map(lambda x: len(f"{x}"), na_action="ignore")
    .max()
    .max(),
    42,
)
coincidencias["hover_info"] = None
coincidencias["hover_info"] = coincidencias.fillna("").apply(
    lambda row: "<br>".join(
        [f"<b>{c:<{flen}}</b>{str(row[c]):>{fd_len}}" for c in hover_cols]
    ),
    axis=1,
)

for i, v in enumerate(sorted(coincidencias["Vía"].unique())):
    if v > 13:
        continue
    use_df = coincidencias[coincidencias["Vía"] == v].copy()
    traces.append(
        go.Scatter(
            x=use_df["Fecha"],
            y=use_df["Coincidencia"],
            # line=dict(width=np.exp(link["Value"]), color="black"),
            line=dict(width=2, color=qualitative.Dark24[i]),
            # fill="toself",
            hoverinfo="text",
            hovertext=use_df["hover_info"],
            # name=link["Source"] + "→" + link["Target"],
            mode="lines+markers",
            name=f"{v}",
            showlegend=True,
            legendgroup=f"{v}",
        )
    )

layout = setLayout("toggleitem", "Coincidencia vías Chamartín", {})
layout["xaxis"]["title"] = "Fecha"
layout["yaxis"]["title"] = "Coincidencia (%)"

fig = go.Figure(data=traces, layout=layout)
# fig.write_html("coincidencia planificación vías chamartin.html")
fig.show()

In [260]:
# df_estaciones

In [26]:
df_estaciones["Código"].unique()

array(['70103', '65000', '72305', '05513', '51050'], dtype=object)

In [30]:
df_estaciones

,Subdirección,Estación,Num. Trenes,Con Vía Programada,% Vía Programada,Registro Vía,% Vía Registrada,Coincide Vía,% Coincide Vía,Fecha,Producto,Código,Nombre,Día,Coincidencia (%),hover_info
23,Centro,70103 - ALCALA DE HENARES,201,201,100,195,97.014925,66,33.846154,2024-02-24,CERCANIAS,70103,ALCALA DE HENARES,2024-02-24,33.846%,<b>Código </b> 7...
54,Este,65000 - VALENCIA-ESTACIO DEL NORD,164,164,100,159,96.95122,23,14.465409,2024-02-24,CERCANIAS,65000,VALENCIA-ESTACIO DEL NORD,2024-02-24,14.465%,<b>Código </b> 6...
110,Noreste,72305 - L'HOSPITALET DE LLOBREGAT,308,308,100,296,96.103896,112,37.837838,2024-02-24,CERCANIAS,72305,L'HOSPITALET DE LLOBREGAT,2024-02-24,37.838%,<b>Código </b> 7...
221,Noroeste,05513 - POLA DE SIERO,31,31,100,8,25.806452,8,100,2024-02-24,CERCANIAS,05513,POLA DE SIERO,2024-02-24,100.000%,<b>Código </b> 0...
369,Sur,51050 - CARTUJA,14,14,100,14,100,6,42.857143,2024-02-24,CERCANIAS,51050,CARTUJA,2024-02-24,42.857%,<b>Código </b> 5...
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
367,Sur,51050 - CARTUJA,10,10,100,10,100,10,100,2024-04-01,CERCANIAS,51050,CARTUJA,2024-04-01,100.000%,<b>Código </b> 5...
108,Noreste,72305 - L'HOSPITALET DE LLOBREGAT,301,301,100,288,95.681063,112,38.888889,2024-04-01,CERCANIAS,72305,L'HOSPITALET DE LLOBREGAT,2024-04-01,38.889%,<b>Código </b> 7...
52,Este,65000 - VALENCIA-ESTACIO DEL NORD,166,166,100,155,93.373494,24,15.483871,2024-04-01,CERCANIAS,65000,VALENCIA-ESTACIO DEL NORD,2024-04-01,15.484%,<b>Código </b> 6...
211,Noroeste,05513 - POLA DE SIERO,48,48,100,44,91.666667,12,27.272727,2024-04-01,CERCANIAS,05513,POLA DE SIERO,2024-04-01,27.273%,<b>Código </b> 0...


In [31]:
traces = []
hover_cols = ["Código", "Nombre", "Día", "Coincidencia (%)"]
flen = max([len(c) for c in hover_cols]) + 2
fd_len = min(
    df_estaciones[hover_cols]
    .fillna("")
    .map(lambda x: len(f"{x}"), na_action="ignore")
    .max()
    .max(),
    42,
)
df_estaciones["hover_info"] = None
df_estaciones["hover_info"] = df_estaciones.fillna("").apply(
    lambda row: "<br>".join(
        [f"<b>{c:<{flen}}</b>{str(row[c]):>{fd_len}}" for c in hover_cols]
    ),
    axis=1,
)

for i, s in enumerate(sorted(df_estaciones["Estación"].unique())):
    use_df = df_estaciones[df_estaciones["Estación"] == s].copy()
    traces.append(
        go.Scatter(
            x=use_df["Fecha"],
            y=use_df["% Coincide Vía"],
            # line=dict(width=np.exp(link["Value"]), color="black"),
            line=dict(width=2, color=qualitative.Dark24[i]),
            # fill="toself",
            hoverinfo="text",
            hovertext=use_df["hover_info"],
            # name=link["Source"] + "→" + link["Target"],
            mode="lines+markers",
            name=f"{s}",
            showlegend=True,
            legendgroup=f"{s}",
        )
    )

layout = setLayout("toggleitem", "Coincidencia vías múltiples estaciones", {})
layout["xaxis"]["title"] = "Fecha"
layout["yaxis"]["title"] = "Coincidencia (%)"

fig = go.Figure(data=traces, layout=layout)
# fig.write_html("coincidencia planificación vías.html")
fig.show()

In [48]:
subset = df_cha.loc[
    (df_cha["FechaOrigen"] == "2024-03-05") & (df_cha["VíaPlanificada"] == 2),
    ["VíaPlanificada", "VíaReal"],
]
subset.apply(lambda x: x["VíaPlanificada"] == x["VíaReal"], axis=1).sum() / len(subset)

0.7703703703703704

In [25]:
t_cols = df_cha.columns
df_split = np.split(
    df_cha.values,
    np.where(
        (~df_cha["VíaPlanificada"].eq(df_cha["VíaPlanificada"].shift()))
        | (~df_cha["VíaPlanificada"].eq(df_cha["VíaPlanificada"].shift()))
    )[0][1:],
)

In [33]:
pd.DataFrame(df_split[5], columns=t_cols)

,Sub.,Tren,FechaOrigen,FechaTeórica,Código,Nombre,VíaPlanificada,VíaReal,FuenteVía
0,1,20205,2024-03-05,2024-03-05,17000,MADRID-CHAMARTIN-CLARA CAMP.,4.0,4.0,CTC
1,1,20207,2024-03-05,2024-03-05,17000,MADRID-CHAMARTIN-CLARA CAMP.,4.0,4.0,CTC
2,1,20209,2024-03-05,2024-03-05,17000,MADRID-CHAMARTIN-CLARA CAMP.,4.0,4.0,CTC
3,1,20211,2024-03-05,2024-03-05,17000,MADRID-CHAMARTIN-CLARA CAMP.,4.0,4.0,CTC
4,1,20213,2024-03-05,2024-03-05,17000,MADRID-CHAMARTIN-CLARA CAMP.,4.0,4.0,CTC
...,...,...,...,...,...,...,...,...,...
131,1,36307,2024-03-05,2024-03-05,17000,MADRID-CHAMARTIN-CLARA CAMP.,4.0,4.0,CTC
132,1,36309,2024-03-05,2024-03-05,17000,MADRID-CHAMARTIN-CLARA CAMP.,4.0,4.0,CTC
133,1,36313,2024-03-05,2024-03-05,17000,MADRID-CHAMARTIN-CLARA CAMP.,4.0,4.0,CTC
134,1,36315,2024-03-05,2024-03-05,17000,MADRID-CHAMARTIN-CLARA CAMP.,4.0,4.0,CTC


In [20]:
df_cha.groupby(["FechaOrigen", "VíaPlanificada"]).agg(list)

Sub.  \
FechaOrigen VíaPlanificada                                                      
2024-03-04  2.0                                                        [1, 1]   
            11.0                                                          [1]   
            13.0                                                       [1, 1]   
2024-03-05  2.0             [1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, ...   
            3.0             [1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, ...   
            4.0             [1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, ...   
            9.0             [1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, ...   
            10.0            [1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, ...   
            11.0            [1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, ...   
            13.0            [1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, ...   

                                                                         Tren  \
FechaOrigen VíaPlanificada                                                      
2024-03-04  2.0                                                [20536, 20538]   
            11.0                                                      [21575]   
            13.0                                               [21172, 21375]   
2024-03-05  2.0             [20204, 20210, 20214, 20218, 20220, 20222, 202...   
            3.0             [20000, 20001, 20002, 20003, 20004, 20005, 200...   
            4.0             [20205, 20207, 20209, 20211, 20213, 20215, 202...   
            9.0             [20600, 20602, 20604, 20606, 20608, 20610, 206...   
            10.0            [19900, 19905, 19906, 19908, 19918, 19919, 199...   
            11.0            [20601, 20603, 20605, 20607, 20609, 20611, 206...   
            13.0            [21100, 21104, 21106, 21108, 21110, 21112, 211...   

                                                                 FechaTeórica  \
FechaOrigen VíaPlanificada                                                      
2024-03-04  2.0                    [2024-03-05 00:00:00, 2024-03-05 00:00:00]   
            11.0                                        [2024-03-05 00:00:00]   
            13.0                   [2024-03-05 00:00:00, 2024-03-05 00:00:00]   
2024-03-05  2.0             [2024-03-05 00:00:00, 2024-03-05 00:00:00, 202...   
            3.0             [2024-03-05 00:00:00, 2024-03-05 00:00:00, 202...   
            4.0             [2024-03-05 00:00:00, 2024-03-05 00:00:00, 202...   
            9.0             [2024-03-05 00:00:00, 2024-03-05 00:00:00, 202...   
            10.0            [2024-03-05 00:00:00, 2024-03-05 00:00:00, 202...   
            11.0            [2024-03-05 00:00:00, 2024-03-05 00:00:00, 202...   
            13.0            [2024-03-05 00:00:00, 2024-03-05 00:00:00, 202...   

                                                                       Código  \
FechaOrigen VíaPlanificada                                                      
2024-03-04  2.0                                                [17000, 17000]   
            11.0                                                      [17000]   
            13.0                                               [17000, 17000]   
2024-03-05  2.0             [17000, 17000, 17000, 17000, 17000, 17000, 170...   
            3.0             [17000, 17000, 17000, 17000, 17000, 17000, 170...   
            4.0             [17000, 17000, 17000, 17000, 17000, 17000, 170...   
            9.0             [17000, 17000, 17000, 17000, 17000, 17000, 170...   
            10.0            [17000, 17000, 17000, 17000, 17000, 17000, 170...   
            11.0            [17000, 17000, 17000, 17000, 17000, 17000, 170...   
            13.0            [17000, 17000, 17000, 17000, 17000, 17000, 170...   

                                                                       Nombre  \
FechaOrigen VíaPlanificada                                                      
2024-03-04  2.0    

In [19]:
df_cha

,Sub.,Tren,FechaOrigen,FechaTeórica,Código,Nombre,VíaPlanificada,VíaReal,FuenteVía
1264,1,20536,2024-03-04,2024-03-05,17000,MADRID-CHAMARTIN-CLARA CAMP.,2.0,2.0,CTC
1265,1,20538,2024-03-04,2024-03-05,17000,MADRID-CHAMARTIN-CLARA CAMP.,2.0,2.0,CTC
1266,1,21172,2024-03-04,2024-03-05,17000,MADRID-CHAMARTIN-CLARA CAMP.,13.0,11.0,CTC
1267,1,21375,2024-03-04,2024-03-05,17000,MADRID-CHAMARTIN-CLARA CAMP.,13.0,9.0,CTC
1268,1,21575,2024-03-04,2024-03-05,17000,MADRID-CHAMARTIN-CLARA CAMP.,11.0,12.0,CTC
...,...,...,...,...,...,...,...,...,...
2117,1,36318,2024-03-05,2024-03-05,17000,MADRID-CHAMARTIN-CLARA CAMP.,2.0,2.0,CTC
2118,1,36320,2024-03-05,2024-03-05,17000,MADRID-CHAMARTIN-CLARA CAMP.,2.0,2.0,CTC
2119,1,36342,2024-03-05,2024-03-05,17000,MADRID-CHAMARTIN-CLARA CAMP.,2.0,2.0,CTC
2120,1,36344,2024-03-05,2024-03-05,17000,MADRID-CHAMARTIN-CLARA CAMP.,2.0,2.0,CTC


In [11]:
df_cha["Vía Real de Estacionamiento"].value_counts()

Vía Real de Estacionamiento
4.0     129
9.0     128
11.0    122
2.0     114
3.0     110
10.0     92
12.0     73
13.0     51
1.0      21
Name: count, dtype: int64